# 🚀 BigQuery Optimization Control Plane: Run On Your Dataset & Organization
### *Interactive Step-by-Step Notebook for Analyzing and Optimizing Any BigQuery Workload*

---

## 🧭 Overview
This notebook guides you through running the **BigQuery Optimization Control Plane** (`bq-optimizer`) on **your own datasets, entire projects, or across your entire GCP Organization**.

It supports three operational scopes:
1. 🎯 **Single Dataset**: Optimize tables and queries in a specific dataset (e.g., `customer360_telco`).
2. 📁 **Project-Wide**: Discover and optimize all tables and datasets within your GCP project.
3. 🏢 **Organization-Wide (`JOBS_BY_ORGANIZATION`)**: Ingest query telemetry across **all projects** in your GCP Organization, correlating org-wide workload patterns against your tables!

```
┌────────────────────────────────────────────────────────┐
│  TIER 1: Central Organization Telemetry Collection     │
│  `INFORMATION_SCHEMA.JOBS_BY_ORGANIZATION` (30+ Projs) │
└────────────────────────────────────────────────────────┘
                           │
                           ▼
┌────────────────────────────────────────────────────────┐
│  TIER 2: Leadership Mapping & Employee Hierarchy       │
│  Correlates user queries with Director & Department    │
└────────────────────────────────────────────────────────┘
                           │
             ┌─────────────┴─────────────┐
             ▼                           ▼
┌───────────────────────────┐ ┌──────────────────────────┐
│  Director Executive View  │ │  Data Platform View      │
│  (e.g., Marty / David S.) │ │  (e.g., Sayali / Chetan) │
│  Spend rollup & team cards│ │  In-place DDL & physical │
│  to prioritize use cases. │ │  clustering/partitioning.│
└───────────────────────────┘ └──────────────────────────┘
```

### 🛡️ Production Safety Guarantees
* **Zero Payload Data Access**: Only reads BigQuery `INFORMATION_SCHEMA` metadata.
* **Human-in-the-Loop Gate**: Nothing executes without your explicit approval.
* **Zero Downtime**: In-place DDL (`ALTER TABLE`) never locks tables or disrupts pipelines.
* **Capped Honest Savings**: Recommendations can never exceed actual table spend.
* **1-Click Rollback**: Any applied change can be rolled back instantly.

---
## ⚙️ Step 0: User Configuration & Environment Setup
Set your Google Cloud Project ID, dataset scope, and organization collection mode below:
* **Single Dataset**: Set `DATASET_ID = "customer360_telco"` to focus recommendations on one dataset.
* **Project-Wide**: Set `DATASET_ID = None` to optimize **all datasets** across your project.
* **Organization-Wide**: Set `RUN_BY_ORGANIZATION = True` to collect and analyze query telemetry across **all projects in your entire GCP Organization** (`INFORMATION_SCHEMA.JOBS_BY_ORGANIZATION`).
* **Leadership & Employee Hierarchy**: Optionally map queries to internal Directors and Departments via `EMPLOYEE_HIERARCHY_TABLE` or use the built-in control plane hierarchy table.

> [!IMPORTANT]
> **IAM Requirement for Organization Scope (`RUN_BY_ORGANIZATION = True`)**:
> BigQuery's `INFORMATION_SCHEMA.JOBS_BY_ORGANIZATION` requires `roles/bigquery.resourceViewer` (specifically `bigquery.jobs.listAll`) granted at the **Organization** or **Folder** level in Google Cloud IAM.

In [ ]:
# 🛠️ Configuration: Set your target project, dataset, and telemetry scope!
PROJECT_ID = "temi-project-408005"      # Central / FinOps GCP Project ID (where optimizer_ops tables live)
DATASET_ID = None                       # Target dataset (e.g. "customer360_telco") OR None for ALL datasets in the project!
LOCATION = "US"                         # BigQuery location (e.g. US, EU, us-central1)
OPS_DATASET = "optimizer_ops"           # Control plane metadata dataset

# 🏢 Organization Scope & Capacity Settings:
RUN_BY_ORGANIZATION = True              # Set to True to collect telemetry across ALL projects in your GCP Organization!
LOOKBACK_DAYS = 30                      # Telemetry lookback window in days (e.g. 7, 14, 30)
RESERVATION_ADMIN_PROJECT = None        # Optional: Central project managing BigQuery slot reservations (or None)

# 🔄 Script Sync Setting:
SYNC_LATEST_SCRIPTS = False             # Set to True to auto-refresh /home/admin_/BQOpt-Optimization-Work with latest files from GCS

# 👔 Enterprise Leadership & Billing Placeholders:
EMPLOYEE_HIERARCHY_TABLE = None         # Set to "enterprise.hr.employee_hierarchy" or None to use optimizer_ops.employee_hierarchy
BILLING_EXPORT_TABLE = None             # Set to "enterprise.billing.gcp_billing_export_v1_xxxx" (optional)

# Summary of active configuration:
target_display = DATASET_ID if DATASET_ID else "(ALL DATASETS IN PROJECT)"
scope_display = "ORGANIZATION-WIDE (JOBS_BY_ORGANIZATION)" if RUN_BY_ORGANIZATION else f"PROJECT-LEVEL ({PROJECT_ID})"

print(f"Targeting Project:     {PROJECT_ID}")
print(f"Targeting Dataset:     {target_display}")
print(f"Telemetry Scope:       {scope_display}")
print(f"Lookback Days:         {LOOKBACK_DAYS}")
print(f"Location:              {LOCATION}")
print(f"Ops Dataset:           {OPS_DATASET}")
if RESERVATION_ADMIN_PROJECT:
    print(f"Reservation Project:   {RESERVATION_ADMIN_PROJECT}")
if EMPLOYEE_HIERARCHY_TABLE:
    print(f"Employee Hierarchy:    {EMPLOYEE_HIERARCHY_TABLE}")
if SYNC_LATEST_SCRIPTS:
    print(f"Script Auto-Sync:      ENABLED (Refreshing from gs://temi-agents/BQOpt-Optimization-Work)")

---
## 📦 Import Libraries & Initialize BigQuery Client
Locates the optimization package, changes to repository root, initializes the BigQuery client, and prepares CLI execution helpers.

In [ ]:
import os
import sys
import json
import subprocess
from google.cloud import bigquery

# 1. Detect environment: Google Colab vs Local Workstation / Jetski IDE
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🚀 Running in Google Colab environment.")
    try:
        from google.colab import auth
        print("[*] Authenticating with Google Cloud...")
        auth.authenticate_user()
        print("✅ Google Cloud user authenticated.")
    except Exception as e:
        print(f"Auth notice: {e}")
    
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive/MyDrive"):
            print("[*] Mounting Google Drive...")
            drive.mount("/content/drive")
            print("✅ Google Drive mounted.")
    except Exception as e:
        print(f"Drive mount notice: {e}")
    
    candidate_roots = [
        "/content/drive/MyDrive/Jetski-Drive-Folder/BQOpt-Optimization-Work",
        "/content/drive/MyDrive/Jetski-Drive-Folder/BQ-Optimization-Control-Tool",
        "/content/drive/MyDrive/BQOpt-Optimization-Work",
        "/content/BQOpt-Optimization-Work",
    ]
else:
    print("💻 Running in local / GCP environment.")
    current_dir = os.getcwd()
    candidate_roots = [
        "/home/admin_/BQOpt-Optimization-Work",
        os.path.expanduser("~/BQOpt-Optimization-Work"),
        current_dir,
        os.path.join(current_dir, "BQOpt-Optimization-Work"),
        "/home/admin_",
        os.path.expanduser("~"),
        os.path.dirname(current_dir),
        "/usr/local/google/home/temiomisore/My-Jetski-Folder/Jetski-Drive-Folder/BQOpt-Optimization-Work",
    ]
    import glob
    candidate_roots.extend(glob.glob("/home/admin_*/BQOpt-Optimization-Work"))
    candidate_roots.extend(glob.glob("/home/admin_*"))

# 2. Locate the repository root containing 'optimizer'
REPO_ROOT = None
for candidate in candidate_roots:
    if candidate and os.path.isdir(os.path.join(candidate, "optimizer")):
        REPO_ROOT = os.path.abspath(candidate)
        break

# Target destination for GCP instance (/home/admin_/BQOpt-Optimization-Work or ~/BQOpt-Optimization-Work)
preferred_dest = "/home/admin_/BQOpt-Optimization-Work" if os.path.isdir("/home/admin_") else os.path.expanduser("~/BQOpt-Optimization-Work")

# If requested by user via SYNC_LATEST_SCRIPTS, or if repo not found, sync directly from GCS
if locals().get("SYNC_LATEST_SCRIPTS", False) or not REPO_ROOT:
    dest_dir = REPO_ROOT if (REPO_ROOT and not locals().get("SYNC_LATEST_SCRIPTS", False)) else preferred_dest
    print(f"[*] Syncing latest scripts from GCS (gs://temi-agents/BQOpt-Optimization-Work) to {dest_dir}...")
    os.makedirs(dest_dir, exist_ok=True)
    try:
        sync_res = subprocess.run([
            "gcloud", "storage", "rsync", "-r", "gs://temi-agents/BQOpt-Optimization-Work", dest_dir
        ], capture_output=True, text=True)
        if os.path.isdir(os.path.join(dest_dir, "optimizer")):
            REPO_ROOT = dest_dir
            print(f"✅ Successfully synced latest scripts to {REPO_ROOT}!")
        else:
            # Fallback to copy if rsync didn't populate subdirectories
            subprocess.run([
                "gcloud", "storage", "cp", "-r", "gs://temi-agents/BQOpt-Optimization-Work", os.path.dirname(dest_dir)
            ], capture_output=True, text=True)
            if os.path.isdir(os.path.join(dest_dir, "optimizer")):
                REPO_ROOT = dest_dir
                print(f"✅ Successfully copied latest scripts to {REPO_ROOT}!")
    except Exception as e:
        print(f"Sync notice: {e}")

if not REPO_ROOT:
    raise FileNotFoundError(
        "Could not find 'optimizer' folder.\n"
        "👉 To download it to your GCP account, run this in your terminal or a notebook cell:\n"
        "   !gcloud storage rsync -r gs://temi-agents/BQOpt-Optimization-Work /home/admin_/BQOpt-Optimization-Work\n"
    )

# CRITICAL: Set sys.path and switch working directory to REPO_ROOT
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

# 3. Prefer local virtualenv python if available
venv_py = os.path.join(REPO_ROOT, ".venv", "bin", "python3")
PYTHON_BIN = venv_py if os.path.isfile(venv_py) else sys.executable

print(f"✅ Working Directory: {os.getcwd()}")
print(f"✅ Repository Root:  {REPO_ROOT}")
print(f"✅ Python Binary:    {PYTHON_BIN}")

# 4. Helper to run optimizer CLI commands with proper cwd and environment
def run_optimizer_cli(args):
    cmd = [PYTHON_BIN, "-m", "optimizer.cli"] + args
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{REPO_ROOT}:{env.get('PYTHONPATH', '')}"
    return subprocess.run(cmd, cwd=REPO_ROOT, env=env, capture_output=True, text=True)

# 5. Optional pandas integration for enhanced table formatting
try:
    import pandas as pd
    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_colwidth", 80)
    pd.set_option("display.width", 1000)
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

# 6. Initialize BigQuery client
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
print(f"✅ Connected to BigQuery as project: {client.project}")

---
## 🔍 Step 1: Inspect Tables in Your Target Dataset / Project
Let's inspect the current state of tables in your dataset or project (row counts, data size in MB, and whether clustering is already enabled).

In [ ]:
# Inspect all tables: works for a single dataset OR project-wide when DATASET_ID = None
if DATASET_ID:
    inspect_sql = f"""
    SELECT 
      t.table_schema AS dataset_id,
      t.table_name,
      t.table_type,
      ROUND(s.total_rows, 0) AS total_rows,
      ROUND(s.total_logical_bytes / POW(1024, 2), 2) AS logical_size_mb,
      ROUND(s.total_physical_bytes / POW(1024, 2), 2) AS physical_size_mb,
      COALESCE((
        SELECT option_value 
        FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLE_OPTIONS` o 
        WHERE o.table_name = t.table_name AND o.option_name = 'clustering_fields'
      ), '(none)') AS current_clustering
    FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES` t
    LEFT JOIN `region-{LOCATION.lower()}`.INFORMATION_SCHEMA.TABLE_STORAGE s
      ON s.project_id = '{PROJECT_ID}'
      AND s.table_schema = '{DATASET_ID}'
      AND s.table_name = t.table_name
    ORDER BY logical_size_mb DESC;
    """
    scope_label = f"dataset '{DATASET_ID}'"
else:
    inspect_sql = f"""
    SELECT 
      t.table_schema AS dataset_id,
      t.table_name,
      t.table_type,
      ROUND(s.total_rows, 0) AS total_rows,
      ROUND(s.total_logical_bytes / POW(1024, 2), 2) AS logical_size_mb,
      ROUND(s.total_physical_bytes / POW(1024, 2), 2) AS physical_size_mb,
      COALESCE((
        SELECT option_value 
        FROM `region-{LOCATION.lower()}`.INFORMATION_SCHEMA.TABLE_OPTIONS o 
        WHERE o.table_schema = t.table_schema AND o.table_name = t.table_name AND o.option_name = 'clustering_fields'
      ), '(none)') AS current_clustering
    FROM `region-{LOCATION.lower()}`.INFORMATION_SCHEMA.TABLES t
    LEFT JOIN `region-{LOCATION.lower()}`.INFORMATION_SCHEMA.TABLE_STORAGE s
      ON s.project_id = '{PROJECT_ID}'
      AND s.table_schema = t.table_schema
      AND s.table_name = t.table_name
    WHERE t.table_schema NOT IN ('{OPS_DATASET}', 'INFORMATION_SCHEMA')
    ORDER BY logical_size_mb DESC;
    """
    scope_label = f"all datasets in project '{PROJECT_ID}'"

try:
    if HAS_PANDAS:
        df_tables = client.query(inspect_sql).to_dataframe()
        if df_tables.empty:
            print(f"ℹ️ No tables found in {scope_label}.")
        else:
            print(f"Found {len(df_tables)} table(s) across {scope_label}:")
            display(df_tables)
    else:
        rows = list(client.query(inspect_sql).result())
        if not rows:
            print(f"ℹ️ No tables found in {scope_label}.")
        else:
            print(f"Found {len(rows)} table(s) across {scope_label}:\n")
            print(f"{'Dataset':<20} | {'Table Name':<28} | {'Type':<12} | {'Rows':<10} | {'Logical (MB)':<14} | {'Clustering':<20}")
            print("-" * 118)
            for r in rows:
                ds = getattr(r, 'dataset_id', DATASET_ID or '')
                print(f"{ds:<20} | {r.table_name:<28} | {r.table_type:<12} | {str(int(r.total_rows or 0)):<10} | {str(r.logical_size_mb or 0.0):<14} | {r.current_clustering:<20}")
except Exception as e:
    print(f"Could not query tables: {e}")

---
## 🏗️ Step 2: Initialize the Optimization Control Plane (`init`)
Creates the `optimizer_ops` dataset if it doesn't already exist.
*Note: If you have already initialized `optimizer_ops`, running this step is completely idempotent and safe.*

In [ ]:
# Run control plane initialization
print(f"[*] Initializing control plane dataset ({OPS_DATASET})...")
try:
    from optimizer.cli import cmd_init
    from optimizer.config import cfg
    c = cfg(project_id=PROJECT_ID, location=LOCATION)
    cmd_init(c)
except Exception as e:
    print(f"Direct Python notice ({e}), running via CLI helper...")
    res = run_optimizer_cli(["init", "-p", PROJECT_ID, "-l", LOCATION])
    print(res.stdout)
    if res.returncode != 0:
        print("Error:", res.stderr)

---
## 📡 Step 3: Collect Telemetry (`collect`)
Extracts query traces and snapshots table metadata into `optimizer_ops`.
* When `RUN_BY_ORGANIZATION = True`, this queries `INFORMATION_SCHEMA.JOBS_BY_ORGANIZATION` to collect query workload traces from **all projects across your entire GCP Organization**!
* When `RUN_BY_ORGANIZATION = False`, this collects telemetry from `INFORMATION_SCHEMA.JOBS` within the current project.
* **Privacy Guarantee**: It **never** reads table data rows. Only query execution metadata, statistics, and column definitions are captured.

In [ ]:
# Collect query & table telemetry (Dataset, Project-wide, or Org-wide)
jobs_view = "JOBS_BY_ORGANIZATION" if RUN_BY_ORGANIZATION else "JOBS"
target_msg = DATASET_ID if DATASET_ID else "(ALL DATASETS IN PROJECT)"
scope_msg = "ORGANIZATION-WIDE" if RUN_BY_ORGANIZATION else f"PROJECT ({PROJECT_ID})"

print(f"[*] Collecting {scope_msg} telemetry for: {target_msg} in project: {PROJECT_ID} (jobs_view={jobs_view})...")

cli_args = [
    "collect",
    "-p", PROJECT_ID,
    "-l", LOCATION,
    "--jobs-view", jobs_view,
    "--lookback-days", str(LOOKBACK_DAYS)
]
if RUN_BY_ORGANIZATION:
    cli_args.append("--org")
if DATASET_ID:
    cli_args.extend(["-d", DATASET_ID])
else:
    cli_args.append("--all-datasets")
if RESERVATION_ADMIN_PROJECT:
    cli_args.extend(["--reservation-admin-project", RESERVATION_ADMIN_PROJECT])

try:
    from optimizer.cli import cmd_collect
    from optimizer.config import cfg
    c = cfg(
        project_id=PROJECT_ID,
        target_dataset=DATASET_ID,
        location=LOCATION,
        jobs_view=jobs_view,
        lookback_days=LOOKBACK_DAYS,
        reservation_admin_project=RESERVATION_ADMIN_PROJECT
    )
    cmd_collect(c)
except Exception as e:
    print(f"Direct Python notice ({e}), running via CLI helper...")
    res = run_optimizer_cli(cli_args)
    print(res.stdout)
    if res.returncode != 0:
        print("Stderr:", res.stderr)

> [!TIP]
> **What if your dataset has had no queries in the last 3 days?**
> The rules engine discovers clustering and partitioning patterns based on real `WHERE` and `JOIN` filters in recent queries.
> If this dataset was newly loaded or idle, run a few representative analytical queries against your tables so BigQuery logs them in `INFORMATION_SCHEMA.JOBS`.
> Run the cell below to inspect the queries captured across your project or organization:

In [ ]:
# Verify recent query activity captured in jobs_events
if DATASET_ID:
    dataset_filter = f"WHERE EXISTS (SELECT 1 FROM UNNEST(referenced_tables) ref WHERE ref.dataset_id = '{DATASET_ID}')"
    scope_desc = f"referencing tables in {DATASET_ID}"
elif RUN_BY_ORGANIZATION:
    dataset_filter = "WHERE project_id IS NOT NULL"
    scope_desc = "across all projects in the GCP Organization"
else:
    dataset_filter = f"WHERE EXISTS (SELECT 1 FROM UNNEST(referenced_tables) ref WHERE ref.dataset_id != '{OPS_DATASET}')"
    scope_desc = f"across all datasets in project '{PROJECT_ID}'"

# If running by organization, display organization-wide breakdowns by project AND Director!
if RUN_BY_ORGANIZATION:
    # 1. Project-level activity breakdown
    org_summary_sql = f"""
    SELECT 
      project_id,
      COUNT(*) AS total_queries_collected,
      ROUND(SUM(total_bytes_billed) / POW(1024, 4), 2) AS total_billed_tb,
      ROUND(SUM(total_slot_ms) / (1000 * 3600), 1) AS total_slot_hours,
      COUNT(DISTINCT user_email) AS active_users
    FROM `{PROJECT_ID}.{OPS_DATASET}.jobs_events`
    GROUP BY project_id
    ORDER BY total_billed_tb DESC
    LIMIT 10;
    """
    try:
        if HAS_PANDAS:
            df_org_summary = client.query(org_summary_sql).to_dataframe()
            if not df_org_summary.empty:
                print("🏢 1. Top Projects by Query Activity (Organization-Wide):")
                display(df_org_summary)
        else:
            org_rows = list(client.query(org_summary_sql).result())
            if org_rows:
                print("🏢 1. Top Projects by Query Activity (Organization-Wide):")
                print(f"{'Project ID':<30} | {'Queries':<10} | {'Billed (TB)':<12} | {'Slot Hours':<12} | {'Users':<8}")
                print("-" * 80)
                for r in org_rows:
                    print(f"{r.project_id:<30} | {r.total_queries_collected:<10} | {str(r.total_billed_tb or 0.0):<12} | {str(r.total_slot_hours or 0.0):<12} | {r.active_users:<8}")
                print()
    except Exception as e:
        print(f"Could not query organization summary: {e}")

    # 2. Director & Leadership Spend Rollup (Employee Hierarchy)
    director_spend_sql = f"""
    SELECT 
      director_name,
      department,
      project_id,
      active_users,
      total_queries,
      total_billed_tb,
      estimated_on_demand_spend_usd,
      total_slot_hours
    FROM `{PROJECT_ID}.{OPS_DATASET}.v_spend_by_director`
    ORDER BY estimated_on_demand_spend_usd DESC
    LIMIT 10;
    """
    try:
        if HAS_PANDAS:
            df_director = client.query(director_spend_sql).to_dataframe()
            if not df_director.empty:
                print("\n👔 2. Leadership & Director Spend Rollup (Executive Governance View):")
                display(df_director)
        else:
            dir_rows = list(client.query(director_spend_sql).result())
            if dir_rows:
                print("\n👔 2. Leadership & Director Spend Rollup (Executive Governance View):")
                print(f"{'Director Name':<32} | {'Department':<20} | {'Project ID':<25} | {'Queries':<8} | {'Est Spend ($)':<14}")
                print("-" * 105)
                for r in dir_rows:
                    print(f"{r.director_name:<32} | {r.department:<20} | {r.project_id:<25} | {r.total_queries:<8} | ${str(r.estimated_on_demand_spend_usd or 0.0):<14}")
                print()
    except Exception as e:
        print(f"Could not query director spend summary: {e}")

# Display recent individual queries
check_jobs_sql = f"""
SELECT 
  project_id,
  creation_time,
  user_email,
  ROUND(total_bytes_billed / POW(1024, 3), 4) AS billed_gb,
  query_preview
FROM `{PROJECT_ID}.{OPS_DATASET}.jobs_events`
{dataset_filter}
ORDER BY creation_time DESC
LIMIT 5;
"""
try:
    if HAS_PANDAS:
        df_recent_jobs = client.query(check_jobs_sql).to_dataframe()
        if df_recent_jobs.empty:
            print(f"ℹ️ Note: No queries found {scope_desc} in the last {LOOKBACK_DAYS} days.")
            print("Storage and metadata rules will still evaluate! For query-based clustering, run queries against your tables.")
        else:
            print(f"\n✅ 3. Recent Individual Queries Captured ({scope_desc}):")
            display(df_recent_jobs)
    else:
        jobs = list(client.query(check_jobs_sql).result())
        if not jobs:
            print(f"ℹ️ Note: No queries found {scope_desc} in the last {LOOKBACK_DAYS} days.")
            print("Storage and metadata rules will still evaluate! For query-based clustering, run queries against your tables.")
        else:
            print(f"\n✅ 3. Recent Individual Queries Captured ({scope_desc}):\n")
            for j in jobs:
                print(f"[{j.creation_time.strftime('%Y-%m-%d %H:%M')}] [{j.project_id}] {j.user_email} | Billed: {j.billed_gb:.4f} GB")
                print(f"  Preview: {j.query_preview[:120]}...\n")
except Exception as e:
    print(f"Could not query jobs_events: {e}")

---
## 🧠 Step 4: Run the Rules Engine & Honest Scoring (`rules`)
Analyzes your dataset against 10+ optimization patterns (Clustering, Partitioning, Storage Billing Models, SQL Anti-Patterns) and calculates honest, capped dollar savings.
Every recommendation is stored in `optimizer_ops.change_sets` in the `PENDING_REVIEW` state.

In [ ]:
# Run rules engine targeted at target dataset or entire project
target_msg = DATASET_ID if DATASET_ID else "(ALL DATASETS IN PROJECT)"
print(f"[*] Running rules engine on: {target_msg}...")

cli_args = ["rules", "-p", PROJECT_ID, "-l", LOCATION]
if DATASET_ID:
    cli_args.extend(["-d", DATASET_ID])
else:
    cli_args.append("--all-datasets")

try:
    from optimizer.cli import cmd_rules
    from optimizer.config import cfg
    c = cfg(project_id=PROJECT_ID, target_dataset=DATASET_ID, location=LOCATION)
    cmd_rules(c)
except Exception as e:
    print(f"Direct Python notice ({e}), running via CLI helper...")
    res = run_optimizer_cli(cli_args)
    print(res.stdout)
    if res.returncode != 0:
        print("Stderr:", res.stderr)

---
## 📋 Step 5: Review Recommendations (The Human-in-the-Loop Gate)
**Nothing is ever applied automatically without your explicit approval.**
Let's inspect the pending recommendations generated by the rules engine:

In [ ]:
# Inspect pending review queue (attributed by Project & Director)
where_clause = f"WHERE target_dataset = '{DATASET_ID}'" if DATASET_ID else "WHERE target_dataset IS NOT NULL"

review_sql = f"""
SELECT 
  change_set_id,
  target_project,
  target_dataset,
  target_table,
  apply_class,
  rule_ids,
  ROUND(gross_monthly_savings_usd, 2) AS projected_monthly_savings_usd,
  state,
  director_name,
  department,
  team_readers_count,
  team_queries_count,
  proposed_change_json
FROM `{PROJECT_ID}.{OPS_DATASET}.v_director_recommendations`
{where_clause}
ORDER BY gross_monthly_savings_usd DESC;
"""

try:
    rows = list(client.query(review_sql).result())
    scope_label = f"for '{DATASET_ID}'" if DATASET_ID else f"across project '{PROJECT_ID}'"
    if not rows:
        print(f"No pending recommendations found {scope_label}. Your tables may already be optimal, or run more query activity.")
    else:
        print(f"Found {len(rows)} pending optimization recommendation(s) {scope_label}:\n")
        if HAS_PANDAS:
            df_pending = client.query(review_sql).to_dataframe()
            display(df_pending[[
                "change_set_id", "target_project", "target_dataset", "target_table",
                "director_name", "department", "projected_monthly_savings_usd", "apply_class"
            ]])
        
        # Print detailed SQL and Director accountability for each card
        for idx, row in enumerate(rows):
            change = json.loads(row["proposed_change_json"]) if row["proposed_change_json"] else {}
            print(f"\n--- Card {idx+1}: {row['target_project']}.{row['target_dataset']}.{row['target_table']} ---")
            print(f"Change Set ID:    {row['change_set_id']}")
            print(f"Action:           {change.get('action')}")
            print(f"Rule IDs:         {row['rule_ids']}")
            print(f"Projected Savings:${row['projected_monthly_savings_usd']:.2f} / mo")
            print(f"Owner / Director: {row['director_name']} ({row['department']})")
            if row.get('team_readers_count'):
                print(f"Impacted Team:    {row['team_readers_count']} users running {row['team_queries_count']} queries")
            if "sql" in change:
                print(f"Proposed SQL:\n{change['sql']}")
except Exception as e:
    print(f"Could not query v_director_recommendations: {e}")

### 🌐 Review Options:
1. **Option A (Web UI - Recommended for Demos)**: Run `python3 review_app/main.py --port 8080` in a terminal and open `http://localhost:8080` to show visual cards, honest scoring, and two-person approval.
2. **Option B (In Notebook)**: Execute the cell below to approve change sets directly via BigQuery SQL!

In [ ]:
# Approve pending change sets directly in BigQuery
where_approve = f"WHERE target_dataset = '{DATASET_ID}' AND state = 'PENDING_REVIEW'" if DATASET_ID else "WHERE state = 'PENDING_REVIEW'"

approve_sql = f"""
UPDATE `{PROJECT_ID}.{OPS_DATASET}.change_sets`
SET state = 'APPROVED',
    approved_by = SESSION_USER(),
    approved_at = CURRENT_TIMESTAMP()
{where_approve};
"""
job = client.query(approve_sql)
job.result()
scope_label = f"for dataset '{DATASET_ID}'" if DATASET_ID else f"across all datasets in '{PROJECT_ID}'"
print(f"✅ Approved change sets {scope_label}. Ready for safe execution!")

---
### 🔍 Step 5.1: Live Approval & Two-Person Governance Verification (For Your Audience)
**🗣️ What you say to your audience:**
> *"When we click 'Approve' in the Web UI or approve via SQL, we can prove to leadership and data owners in real time that every action is cryptographically recorded in BigQuery's audit log.*
> *Notice how Class 3 structural changes require **Two-Person Approval**: the first signature records the Data Owner sign-off, leaving the state as `PENDING_REVIEW` until the Platform Admin provides the second sign-off."*

Run the cell below to inspect the live audit log in `change_sets`:

In [ ]:
# Live Approval Status & Signature Verification SQL
verify_approvals_sql = f"""
SELECT 
  c.change_set_id,
  c.target_dataset,
  c.target_table,
  c.apply_class,
  c.state,
  COALESCE(ARRAY_LENGTH(c.approvals), 0) AS signature_count,
  ARRAY_TO_STRING(ARRAY(
    SELECT FORMAT('%s (%s)', app.principal, app.role) FROM UNNEST(c.approvals) app
  ), ', ') AS approver_signatures,
  (SELECT note FROM UNNEST(c.state_history) ORDER BY `at` DESC LIMIT 1) AS latest_audit_trail_note
FROM `{PROJECT_ID}.{OPS_DATASET}.change_sets` c
WHERE c.state IN ('PENDING_REVIEW', 'APPROVED')
ORDER BY c.created_at DESC;
"""
try:
    if HAS_PANDAS:
        df_app = client.query(verify_approvals_sql).to_dataframe()
        print("📋 Current Approval Signatures & Audit Trail in BigQuery:")
        display(df_app)
    else:
        rows = list(client.query(verify_approvals_sql).result())
        print("📋 Current Approval Signatures & Audit Trail in BigQuery:\n")
        print(f"{'Target Table':<28} | {'Class':<5} | {'State':<15} | {'Signatures':<10} | {'Approvers'}")
        print("-" * 105)
        for r in rows:
            tbl = f"{r.target_dataset}.{r.target_table}"
            print(f"{tbl:<28} | {r.apply_class:<5} | {r.state:<15} | {r.signature_count:<10} | {r.approver_signatures}")
except Exception as e:
    print(f"Could not verify approvals: {e}")

---
## ⚡ Step 6: Safe Execution with Guardrails (`execute`)
Now we apply ONLY the recommendations that were explicitly approved.
* **Class 1 In-Place DDL**: Executes `ALTER TABLE ... SET OPTIONS(...)` directly on BigQuery.
  * **Zero downtime**: Readers and writers are never blocked.
  * Freezes pre-apply query performance baseline for savings verification.
  * Moves state to `APPLIED`, then `VERIFYING`.

In [ ]:
# Run the executor to apply approved optimizations with zero downtime
target_msg = DATASET_ID if DATASET_ID else "(ALL APPROVED CHANGES IN PROJECT)"
print(f"[*] Executing approved changes for: {target_msg}...")

cli_args = ["execute", "-p", PROJECT_ID, "-l", LOCATION]
if DATASET_ID:
    cli_args.extend(["-d", DATASET_ID])
else:
    cli_args.append("--all-datasets")

try:
    from optimizer.cli import cmd_execute
    from optimizer.config import cfg
    c = cfg(project_id=PROJECT_ID, target_dataset=DATASET_ID, location=LOCATION)
    cmd_execute(c)
except Exception as e:
    print(f"Direct Python notice ({e}), running via CLI helper...")
    res = run_optimizer_cli(cli_args)
    print(res.stdout)
    if res.returncode != 0:
        print("Stderr:", res.stderr)

---
### 🔍 Step 6.1: Live DDL Execution & Applied State Verification (For Your Audience)
**🗣️ What you say to your audience:**
> *"Now let's check BigQuery to verify that our approved optimizations physically executed on the database with zero downtime.*
> *We verify two things: first, that `change_sets` moved from `APPROVED` to `APPLIED` / `VERIFYING`; second, that the target tables in `INFORMATION_SCHEMA.TABLE_OPTIONS` now actually have the clustering and partition filters applied!"*

In [ ]:
# Verify applied changes in change_sets and physical table options in BigQuery
verify_applied_sql = f"""
SELECT 
  c.change_set_id,
  c.target_dataset,
  c.target_table,
  c.apply_class,
  c.state,
  c.applied_at,
  (SELECT note FROM UNNEST(c.state_history) WHERE state = 'APPLIED' LIMIT 1) AS execution_note
FROM `{PROJECT_ID}.{OPS_DATASET}.change_sets` c
WHERE c.state IN ('APPLIED', 'VERIFYING')
ORDER BY c.applied_at DESC;
"""

table_options_sql = f"""
SELECT 
  table_schema AS dataset_id,
  table_name,
  option_name,
  option_value
FROM `region-{LOCATION.lower()}`.INFORMATION_SCHEMA.TABLE_OPTIONS
WHERE option_name IN ('clustering_fields', 'require_partition_filter', 'partition_expiration_days')
  AND table_schema != '{OPS_DATASET}'
ORDER BY table_schema, table_name;
"""

try:
    if HAS_PANDAS:
        print("⚡ Applied Change Sets Status in Control Plane:")
        display(client.query(verify_applied_sql).to_dataframe())
        print("\n🏛️ Active Table Options in BigQuery (Proving In-Place DDL Applied):")
        display(client.query(table_options_sql).to_dataframe())
    else:
        print("⚡ Applied Change Sets Status in Control Plane:")
        for r in client.query(verify_applied_sql).result():
            print(f"Table: {r.target_dataset}.{r.target_table} | State: {r.state} | Applied: {r.applied_at}")
        print("\n🏛️ Active Table Options in BigQuery:")
        for r in client.query(table_options_sql).result():
            print(f"Table: {r.dataset_id}.{r.table_name} | {r.option_name} = {r.option_value}")
except Exception as e:
    print(f"Could not verify execution: {e}")

---
## 🧾 Step 7: Verify Realized Savings (The CFO Receipt)
The Verifier measures post-execution query scans against the pre-execution baseline:
* **Regression Watchdog**: Alerts if queries scan >15% more data than before.
* **CFO Receipt**: Calculates realized dollar savings and stores them in `v_receipts`.

In [ ]:
# Run verifier
print("[*] Running verifier...")
try:
    from optimizer.cli import cmd_verify
    from optimizer.config import cfg
    c = cfg(project_id=PROJECT_ID, location=LOCATION)
    cmd_verify(c)
except Exception as e:
    print(f"Direct Python notice ({e}), running via CLI helper...")
    res = run_optimizer_cli(["verify", "-p", PROJECT_ID, "-l", LOCATION])
    print(res.stdout)
    if res.returncode != 0:
        print("Stderr:", res.stderr)

# Display CFO proof receipts
where_receipts = f"WHERE target_dataset = '{DATASET_ID}'" if DATASET_ID else "WHERE target_dataset IS NOT NULL"

receipts_sql = f"""
SELECT 
  change_set_id,
  target_dataset,
  target_table,
  predicted_usd,
  realized_usd,
  realized_over_predicted,
  state,
  applied_at
FROM `{PROJECT_ID}.{OPS_DATASET}.v_receipts`
{where_receipts}
ORDER BY applied_at DESC;
"""
try:
    if HAS_PANDAS:
        df_receipts = client.query(receipts_sql).to_dataframe()
        if df_receipts.empty:
            print("No receipts recorded yet (queries need to execute over the verification window).")
        else:
            scope_desc = f"for {DATASET_ID}" if DATASET_ID else f"across {PROJECT_ID}"
            print(f"✅ Verified Optimization Receipts {scope_desc}:\n")
            display(df_receipts)
    else:
        receipts = list(client.query(receipts_sql).result())
        if not receipts:
            print("No receipts recorded yet (queries need to execute over the verification window).")
        else:
            scope_desc = f"for {DATASET_ID}" if DATASET_ID else f"across {PROJECT_ID}"
            print(f"✅ Verified Optimization Receipts {scope_desc}:\n")
            for r in receipts:
                print(f"Table: {r.target_dataset}.{r.target_table} | Predicted: ${r.predicted_usd or 0.0:.2f}/mo | Realized: ${r.realized_usd or 0.0:.2f}/mo | State: {r.state}")
except Exception as e:
    print("Error querying v_receipts:", e)

---
## ⏪ Step 8: Safety Net & 1-Click Rollback Demo
If an engineer or downstream team ever reports an unexpected issue, any optimization can be rolled back in under 2 seconds.
* **Class 1**: Reverts/drops the clustering metadata options in place.
* **Class 3**: Swaps the untouched zero-copy clone table back into production.
* **Audit Trail**: Records operator email, timestamp, and audit reason in `state_history`.

In [ ]:
# 1-Click Rollback: Automatically selects the most recently applied change set, or specify an ID
CHANGE_SET_TO_ROLLBACK = None  # Leave None to rollback the most recent change, or paste a change_set_id string

if not CHANGE_SET_TO_ROLLBACK:
    find_latest_sql = f"""
    SELECT change_set_id, target_dataset, target_table, apply_class
    FROM `{PROJECT_ID}.{OPS_DATASET}.change_sets`
    WHERE state IN ('APPLIED', 'VERIFYING')
    ORDER BY applied_at DESC
    LIMIT 1;
    """
    rows = list(client.query(find_latest_sql).result())
    if rows:
        CHANGE_SET_TO_ROLLBACK = rows[0].change_set_id
        print(f"[*] Target for rollback: {CHANGE_SET_TO_ROLLBACK} ({rows[0].target_dataset}.{rows[0].target_table})")
    else:
        print("ℹ️ No change sets currently in APPLIED or VERIFYING state to roll back.")
        print("If you want to test rollback, execute an approved change set in Step 6 first!")

if CHANGE_SET_TO_ROLLBACK:
    print(f"[*] Initiating 1-Click Rollback on {CHANGE_SET_TO_ROLLBACK}...")
    try:
        from optimizer.cli import cmd_rollback
        from optimizer.config import cfg
        c = cfg(project_id=PROJECT_ID, location=LOCATION)
        cmd_rollback(c, CHANGE_SET_TO_ROLLBACK, reason="Audience Demo: Verified 1-Click Rollback")
        print(f"✅ Rollback of {CHANGE_SET_TO_ROLLBACK} completed successfully!")
    except Exception as e:
        print(f"Direct Python notice ({e}), running via CLI helper...")
        res = run_optimizer_cli(["rollback", "--id", CHANGE_SET_TO_ROLLBACK, "--reason", "Audience Demo: Verified 1-Click Rollback"])
        print(res.stdout)
        if res.returncode != 0:
            print("Stderr:", res.stderr)

---
### 🔍 Step 8.1: Live Rollback Verification SQL (Prove It to Your Audience)
**🗣️ What you say to your audience:**
> *"Now let's prove in BigQuery that the rollback worked!*
> *We run this query to show that the change set immediately moved to `ROLLED_BACK`, capturing who requested the rollback, when it occurred, and the exact business reason.*
> *We also re-inspect the table options to show that the table was restored back to its original state."*

In [ ]:
# Live Rollback Verification SQL
verify_rollback_sql = f"""
SELECT 
  c.change_set_id,
  c.target_dataset,
  c.target_table,
  c.apply_class,
  c.state,
  sh.actor AS rolled_back_by,
  sh.`at` AS rolled_back_at,
  sh.note AS rollback_reason
FROM `{PROJECT_ID}.{OPS_DATASET}.change_sets` c,
UNNEST(state_history) AS sh
WHERE c.state = 'ROLLED_BACK'
  AND sh.state = 'ROLLED_BACK'
ORDER BY rolled_back_at DESC
LIMIT 5;
"""

table_options_sql = f"""
SELECT 
  table_schema AS dataset_id,
  table_name,
  option_name,
  option_value
FROM `region-{LOCATION.lower()}`.INFORMATION_SCHEMA.TABLE_OPTIONS
WHERE option_name IN ('clustering_fields', 'require_partition_filter', 'partition_expiration_days')
  AND table_schema != '{OPS_DATASET}'
ORDER BY table_schema, table_name;
"""

try:
    if HAS_PANDAS:
        df_rb = client.query(verify_rollback_sql).to_dataframe()
        if df_rb.empty:
            print("No rolled back change sets found yet.")
        else:
            print("⏪ Verified Rollback Audit Trail in BigQuery:")
            display(df_rb)
            print("\n🏛️ Active Table Options in BigQuery (Proving Options Were Reverted):")
            display(client.query(table_options_sql).to_dataframe())
    else:
        rows = list(client.query(verify_rollback_sql).result())
        if not rows:
            print("No rolled back change sets found yet.")
        else:
            print("⏪ Verified Rollback Audit Trail in BigQuery:\n")
            for r in rows:
                print(f"Table: {r.target_dataset}.{r.target_table} | State: {r.state} | By: {r.rolled_back_by} | Reason: {r.rollback_reason}")
            print("\n🏛️ Active Table Options in BigQuery:")
            for r in client.query(table_options_sql).result():
                print(f"Table: {r.dataset_id}.{r.table_name} | {r.option_name} = {r.option_value}")
except Exception as e:
    print(f"Could not verify rollback: {e}")